# HUMMBL Governance-as-a-Service: Interactive Demo

---

**AI agents are powerful. Ungoverned AI agents are dangerous.**

When you deploy multiple AI agents that can delegate tasks to each other, call external APIs, and make decisions autonomously, you need governance primitives that are:

- **Cryptographically signed** -- every delegation is tamper-proof
- **Automatically bounded** -- agents cannot exceed their authority
- **Self-healing** -- failures are contained, not cascaded
- **Fully auditable** -- every action is logged immutably

HUMMBL GaaS provides five production-grade governance primitives, all implemented in pure Python stdlib with zero third-party dependencies:

| Primitive | Purpose |
|---|---|
| **Delegation Tokens** | HMAC-SHA256 signed capability tokens with expiry and scope |
| **Circuit Breaker** | Automatic failure detection and recovery (CLOSED/OPEN/HALF_OPEN) |
| **Kill Switch** | Graduated emergency halt (4 modes) |
| **Governance Bus** | Append-only JSONL audit log with retention |
| **Delegation Chains** | Bounded subdelegation with depth enforcement |

This notebook is a **5-minute interactive demo**. Run each cell to see governance in action.

> Built by [HUMMBL](https://hummbl.io) | Zero dependencies | Production-tested with 5,000+ tests

## Cell 1: Setup

This cell inlines the core governance classes so the notebook runs standalone in Google Colab -- no pip install required. In production, these are separate modules in the `hummbl-governance` package.

In [ ]:
# =============================================================================
# HUMMBL GaaS Core -- Inlined for standalone Colab execution
# In production: pip install hummbl-governance
# =============================================================================

from __future__ import annotations

import hashlib
import hmac
import json
import os
import threading
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from enum import Enum, auto
from pathlib import Path
from typing import Any, Callable, Iterator, Literal, Optional
import tempfile

# ---------------------------------------------------------------------------
# Enable IDP (Intelligent Delegation Profile) for this demo
# ---------------------------------------------------------------------------
os.environ["ENABLE_IDP"] = "true"
os.environ["DCT_SECRET"] = "demo-secret-key-do-not-use-in-production"

# Create a temp directory for governance logs
_DEMO_DIR = Path(tempfile.mkdtemp(prefix="gaas_demo_"))
print(f"Demo workspace: {_DEMO_DIR}")


# ===== DELEGATION TOKENS =====================================================

def _compute_signature(data: dict[str, Any], secret: bytes) -> str:
    """Compute HMAC-SHA256 signature for token data."""
    canonical = json.dumps(data, sort_keys=True, separators=(",", ":"))
    return hmac.new(secret, canonical.encode("utf-8"), hashlib.sha256).hexdigest()


@dataclass(frozen=True)
class ResourceSelector:
    """Resource selector specifying accessible resources."""
    resource_type: str
    resource_id: str = "*"
    constraints: dict[str, Any] = field(default_factory=dict)


@dataclass(frozen=True)
class TokenBinding:
    """Binding linking a token to a specific task and contract."""
    task_id: str
    contract_id: str


@dataclass(frozen=True)
class DelegationCapabilityToken:
    """HMAC-SHA256 signed delegation capability token."""
    token_id: str
    issuer: str
    subject: str
    resource_selectors: tuple[ResourceSelector, ...] = field(default_factory=tuple)
    ops_allowed: tuple[str, ...] = field(default_factory=tuple)
    expiry: str | None = None
    binding: TokenBinding | None = None
    signature: str = ""

    def to_dict(self) -> dict[str, Any]:
        return {
            "token_id": self.token_id,
            "issuer": self.issuer,
            "subject": self.subject,
            "resource_selectors": [
                {"resource_type": r.resource_type, "resource_id": r.resource_id,
                 "constraints": r.constraints} for r in self.resource_selectors
            ],
            "ops_allowed": list(self.ops_allowed),
            "expiry": self.expiry,
            "binding": {"task_id": self.binding.task_id,
                        "contract_id": self.binding.contract_id} if self.binding else None,
        }

    def verify_signature(self, secret: bytes) -> bool:
        return hmac.compare_digest(self.signature, _compute_signature(self.to_dict(), secret))

    def is_expired(self) -> bool:
        if self.expiry is None:
            return False
        try:
            expiry_dt = datetime.fromisoformat(self.expiry.replace("Z", "+00:00"))
            return datetime.now(timezone.utc) > expiry_dt
        except (ValueError, TypeError):
            return True


class DelegationTokenManager:
    """Manager for creating and validating Delegation Capability Tokens."""

    def __init__(self, secret: bytes):
        self._secret = secret

    def create_token(
        self, issuer: str, subject: str, ops_allowed: list[str],
        binding: TokenBinding, resource_selectors: list[ResourceSelector] | None = None,
        expiry_minutes: int | None = 120,
    ) -> DelegationCapabilityToken:
        expiry = None
        if expiry_minutes is not None:
            expiry_dt = datetime.now(timezone.utc) + timedelta(minutes=expiry_minutes)
            expiry = expiry_dt.isoformat().replace("+00:00", "Z")

        token = DelegationCapabilityToken(
            token_id=str(uuid.uuid4()), issuer=issuer, subject=subject,
            resource_selectors=tuple(resource_selectors or []),
            ops_allowed=tuple(ops_allowed), expiry=expiry, binding=binding,
        )
        sig = _compute_signature(token.to_dict(), self._secret)
        return DelegationCapabilityToken(
            token_id=token.token_id, issuer=token.issuer, subject=token.subject,
            resource_selectors=token.resource_selectors, ops_allowed=token.ops_allowed,
            expiry=token.expiry, binding=token.binding, signature=sig,
        )

    def validate_token(self, token: DelegationCapabilityToken) -> tuple[bool, str | None]:
        if not token.verify_signature(self._secret):
            return False, "IDP_E_TOKEN_INVALID"
        if token.is_expired():
            return False, "IDP_E_TOKEN_EXPIRED"
        return True, None

    def check_least_privilege(self, token: DelegationCapabilityToken, requested_op: str) -> tuple[bool, str | None]:
        if requested_op not in token.ops_allowed:
            return False, "IDP_E_DCT_VIOLATION"
        return True, None


# ===== CIRCUIT BREAKER ========================================================

class CircuitBreakerState(Enum):
    CLOSED = auto()
    OPEN = auto()
    HALF_OPEN = auto()


class CircuitBreakerOpen(Exception):
    def __init__(self, message="Circuit breaker is open", *, failure_count=0, recovery_timeout=0.0):
        self.failure_count = failure_count
        self.recovery_timeout = recovery_timeout
        super().__init__(message)


class CircuitBreaker:
    """Automatic failure detection and recovery."""

    def __init__(self, failure_threshold: int = 5, recovery_timeout: float = 30.0,
                 on_state_change: Optional[Callable] = None):
        self._failure_threshold = failure_threshold
        self._recovery_timeout = recovery_timeout
        self._on_state_change = on_state_change
        self._state = CircuitBreakerState.CLOSED
        self._failure_count = 0
        self._success_count = 0
        self._last_failure_time: float | None = None
        self._half_open_probe_in_flight = False
        self._lock = threading.Lock()

    @property
    def state(self) -> CircuitBreakerState:
        with self._lock:
            return self._effective_state()

    @property
    def failure_count(self) -> int:
        return self._failure_count

    def _effective_state(self) -> CircuitBreakerState:
        if self._state == CircuitBreakerState.OPEN and self._last_failure_time is not None:
            if time.monotonic() - self._last_failure_time >= self._recovery_timeout:
                return CircuitBreakerState.HALF_OPEN
        return self._state

    def call(self, fn: Callable[..., Any], *args, **kwargs) -> Any:
        with self._lock:
            effective = self._effective_state()
            if effective == CircuitBreakerState.OPEN:
                raise CircuitBreakerOpen(
                    f"Circuit breaker OPEN (failures={self._failure_count})",
                    failure_count=self._failure_count, recovery_timeout=self._recovery_timeout,
                )
            if effective == CircuitBreakerState.HALF_OPEN:
                if self._state == CircuitBreakerState.OPEN:
                    old = self._state
                    self._state = CircuitBreakerState.HALF_OPEN
                    self._fire_callback(old, CircuitBreakerState.HALF_OPEN)
                if self._half_open_probe_in_flight:
                    raise CircuitBreakerOpen("Probe already in progress")
                self._half_open_probe_in_flight = True

        try:
            result = fn(*args, **kwargs)
        except Exception:
            with self._lock:
                self._failure_count += 1
                self._last_failure_time = time.monotonic()
                if self._state == CircuitBreakerState.HALF_OPEN:
                    self._half_open_probe_in_flight = False
                    old = self._state; self._state = CircuitBreakerState.OPEN
                    self._fire_callback(old, CircuitBreakerState.OPEN)
                elif self._state == CircuitBreakerState.CLOSED and self._failure_count >= self._failure_threshold:
                    old = self._state; self._state = CircuitBreakerState.OPEN
                    self._fire_callback(old, CircuitBreakerState.OPEN)
            raise

        with self._lock:
            self._success_count += 1
            if self._state == CircuitBreakerState.HALF_OPEN:
                self._half_open_probe_in_flight = False
                self._failure_count = 0
                self._last_failure_time = None
                old = self._state; self._state = CircuitBreakerState.CLOSED
                self._fire_callback(old, CircuitBreakerState.CLOSED)
            elif self._state == CircuitBreakerState.CLOSED:
                self._failure_count = 0
        return result

    def reset(self):
        with self._lock:
            old = self._state
            self._failure_count = 0; self._success_count = 0
            self._last_failure_time = None; self._half_open_probe_in_flight = False
            if self._state != CircuitBreakerState.CLOSED:
                self._state = CircuitBreakerState.CLOSED
                self._fire_callback(old, CircuitBreakerState.CLOSED)

    def _fire_callback(self, old, new):
        if self._on_state_change:
            try: self._on_state_change(old, new)
            except Exception: pass


# ===== KILL SWITCH =============================================================

class KillSwitchMode(Enum):
    DISENGAGED = auto()
    HALT_NONCRITICAL = auto()
    HALT_ALL = auto()
    EMERGENCY = auto()


@dataclass(frozen=True)
class KillSwitchEvent:
    timestamp: str
    mode: KillSwitchMode
    reason: str
    triggered_by: str
    affected_tasks: int = 0


class KillSwitchCore:
    """Emergency halt system with graduated response."""

    CRITICAL_TASKS: frozenset[str] = frozenset([
        "safety_monitoring", "data_persistence", "audit_logging",
        "kill_switch_itself", "cost_tracking",
    ])

    def __init__(self):
        self._mode = KillSwitchMode.DISENGAGED
        self._history: list[KillSwitchEvent] = []
        self._subscribers: list[Callable] = []

    @property
    def mode(self) -> KillSwitchMode:
        return self._mode

    @property
    def engaged(self) -> bool:
        return self._mode != KillSwitchMode.DISENGAGED

    def subscribe(self, callback: Callable):
        self._subscribers.append(callback)

    def engage(self, mode: KillSwitchMode, reason: str, triggered_by: str,
               affected_tasks: int = 0) -> KillSwitchEvent:
        if mode == KillSwitchMode.DISENGAGED:
            raise ValueError("Use disengage() instead")
        self._mode = mode
        event = KillSwitchEvent(
            timestamp=datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
            mode=mode, reason=reason, triggered_by=triggered_by,
            affected_tasks=affected_tasks,
        )
        self._history.append(event)
        for cb in self._subscribers:
            try: cb(event)
            except Exception: pass
        return event

    def disengage(self, triggered_by: str, reason: str | None = None) -> KillSwitchEvent:
        prev = self._mode
        self._mode = KillSwitchMode.DISENGAGED
        event = KillSwitchEvent(
            timestamp=datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
            mode=KillSwitchMode.DISENGAGED,
            reason=reason or f"Disengaged from {prev.name}",
            triggered_by=triggered_by,
        )
        self._history.append(event)
        for cb in self._subscribers:
            try: cb(event)
            except Exception: pass
        return event

    def check_task_allowed(self, task_type: str) -> dict[str, Any]:
        is_critical = task_type in self.CRITICAL_TASKS
        if self._mode == KillSwitchMode.DISENGAGED:
            return {"allowed": True, "action": "allow"}
        if self._mode == KillSwitchMode.HALT_NONCRITICAL:
            if is_critical:
                return {"allowed": True, "action": "allow", "note": "critical task exempted"}
            return {"allowed": False, "action": "queue",
                    "reason": f"Kill switch {self._mode.name}: {task_type} queued"}
        if self._mode in (KillSwitchMode.HALT_ALL, KillSwitchMode.EMERGENCY):
            if is_critical and self._mode == KillSwitchMode.HALT_ALL:
                return {"allowed": True, "action": "allow", "note": "critical only"}
            return {"allowed": False, "action": "block",
                    "reason": f"Kill switch {self._mode.name}: {task_type} blocked"}
        return {"allowed": False, "action": "block", "reason": "Unknown state"}

    def get_history(self) -> list[KillSwitchEvent]:
        return self._history.copy()


# ===== GOVERNANCE BUS ==========================================================

@dataclass(frozen=True)
class GovernanceEntry:
    """Single entry in the governance audit log."""
    timestamp: str
    entry_id: str
    intent_id: str
    task_id: str
    tuple_type: str  # DCTX, CONTRACT, EVIDENCE, ATTEST, DCT, SYSTEM
    tuple_data: dict[str, Any]
    signature: str | None = None

    def to_jsonl(self) -> str:
        return json.dumps({
            "timestamp": self.timestamp, "entry_id": self.entry_id,
            "intent_id": self.intent_id, "task_id": self.task_id,
            "tuple_type": self.tuple_type, "tuple_data": self.tuple_data,
            "signature": self.signature,
        }, sort_keys=True, separators=(",", ":"))


class GovernanceBus:
    """Append-only governance audit log."""

    def __init__(self, base_dir: Path):
        self._base_dir = base_dir
        self._base_dir.mkdir(parents=True, exist_ok=True)
        self._entries: list[GovernanceEntry] = []
        self._lock = threading.Lock()

    def append(self, intent_id: str, task_id: str, tuple_type: str,
               tuple_data: dict[str, Any], signature: str | None = None) -> GovernanceEntry:
        entry = GovernanceEntry(
            timestamp=datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
            entry_id=str(uuid.uuid4()), intent_id=intent_id, task_id=task_id,
            tuple_type=tuple_type, tuple_data=tuple_data,
            signature=signature or self._sign(intent_id, task_id, tuple_type, tuple_data),
        )
        with self._lock:
            self._entries.append(entry)
            log_file = self._base_dir / f"governance-{datetime.now(timezone.utc).strftime('%Y-%m-%d')}.jsonl"
            with open(log_file, "a", encoding="utf-8") as f:
                f.write(entry.to_jsonl() + "\n")
        return entry

    def _sign(self, intent_id, task_id, tuple_type, tuple_data) -> str:
        payload = json.dumps({"intent_id": intent_id, "task_id": task_id,
                              "tuple_type": tuple_type, "tuple_data": tuple_data},
                             sort_keys=True, separators=(",", ":"))
        secret = os.environ.get("DCT_SECRET", "ephemeral").encode("utf-8")
        return hmac.new(secret, payload.encode("utf-8"), hashlib.sha256).hexdigest()

    def query(self, intent_id: str | None = None, task_id: str | None = None,
              tuple_type: str | None = None, agent: str | None = None) -> list[GovernanceEntry]:
        results = []
        for e in self._entries:
            if intent_id and e.intent_id != intent_id:
                continue
            if task_id and e.task_id != task_id:
                continue
            if tuple_type and e.tuple_type != tuple_type:
                continue
            if agent and e.tuple_data.get("agent") != agent:
                continue
            results.append(e)
        return results

    @property
    def entries(self) -> list[GovernanceEntry]:
        return self._entries.copy()

    @property
    def count(self) -> int:
        return len(self._entries)


# ===== DELEGATION CONTEXT ======================================================

DEFAULT_MAX_CHAIN_DEPTH = 3
DEFAULT_MAX_REPLANS = 2

DCTXStatus = Literal[
    "PROPOSED", "ISSUED", "RUNNING", "EVIDENCE_READY",
    "VERIFIED", "REPLANNED", "FAILED",
]


@dataclass
class DelegationBudget:
    max_tokens: int = 0
    max_cost_usd: float = 0.0
    max_wall_time_seconds: int = 0

    def is_exceeded(self, tokens=0, cost=0.0, seconds=0) -> bool:
        if self.max_tokens > 0 and tokens > self.max_tokens:
            return True
        if self.max_cost_usd > 0 and cost > self.max_cost_usd:
            return True
        if self.max_wall_time_seconds > 0 and seconds > self.max_wall_time_seconds:
            return True
        return False


@dataclass
class DelegationContext:
    """Delegation Context with chain depth enforcement."""
    intent_id: str
    task_id: str
    delegator_id: str
    delegatee_id: str
    contract_id: str
    parent_task_id: str | None = None
    chain_depth: int = 0
    budget: DelegationBudget = field(default_factory=DelegationBudget)
    status: str = "PROPOSED"  # DCTXStatus
    replan_count: int = 0
    metadata: dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        if self.chain_depth > DEFAULT_MAX_CHAIN_DEPTH:
            raise ValueError(f"IDP_E_DEPTH_EXCEEDED: chain_depth {self.chain_depth} > max {DEFAULT_MAX_CHAIN_DEPTH}")

    def transition(self, new_status: str) -> tuple[bool, str | None]:
        valid = {
            "PROPOSED": ["ISSUED"], "ISSUED": ["RUNNING", "FAILED"],
            "RUNNING": ["EVIDENCE_READY", "FAILED"],
            "EVIDENCE_READY": ["VERIFIED", "REPLANNED"],
            "REPLANNED": ["PROPOSED", "FAILED"], "VERIFIED": [], "FAILED": [],
        }
        if new_status not in valid.get(self.status, []):
            return False, "IDP_E_INVALID_STATE_TRANSITION"
        if self.status == "EVIDENCE_READY" and new_status == "REPLANNED":
            if self.replan_count >= DEFAULT_MAX_REPLANS:
                return False, "IDP_E_REPLAN_LIMIT"
        if new_status == "REPLANNED":
            self.replan_count += 1
        self.status = new_status
        return True, None

    def can_subdelegate(self, max_depth=DEFAULT_MAX_CHAIN_DEPTH) -> tuple[bool, str | None]:
        if self.chain_depth + 1 > max_depth:
            return False, "IDP_E_DEPTH_EXCEEDED"
        return True, None

    def create_child(self, delegatee_id: str, contract_id: str,
                     budget: DelegationBudget | None = None) -> tuple['DelegationContext | None', str | None]:
        can_sub, error = self.can_subdelegate()
        if not can_sub:
            return None, error
        child = DelegationContext(
            intent_id=self.intent_id, task_id=str(uuid.uuid4()),
            parent_task_id=self.task_id, delegator_id=self.delegatee_id,
            delegatee_id=delegatee_id, contract_id=contract_id,
            chain_depth=self.chain_depth + 1,
            budget=budget or DelegationBudget(
                max_tokens=self.budget.max_tokens, max_cost_usd=self.budget.max_cost_usd,
                max_wall_time_seconds=self.budget.max_wall_time_seconds,
            ),
        )
        return child, None


# ===== SHARED INSTANCES ========================================================

SECRET = os.environ["DCT_SECRET"].encode("utf-8")
token_mgr = DelegationTokenManager(secret=SECRET)
audit_log = GovernanceBus(base_dir=_DEMO_DIR / "governance")
kill_switch = KillSwitchCore()

# Wire kill switch events into governance log
kill_switch.subscribe(lambda evt: audit_log.append(
    intent_id="system", task_id="kill_switch",
    tuple_type="SYSTEM",
    tuple_data={"action": "kill_switch_transition", "mode": evt.mode.name,
                "reason": evt.reason, "triggered_by": evt.triggered_by},
))

print("[OK] All governance primitives initialized.")
print(f"[OK] Token manager ready (HMAC-SHA256)")
print(f"[OK] Governance audit log at {_DEMO_DIR / 'governance'}")
print(f"[OK] Kill switch DISENGAGED")
print()
print("Ready to demo. Run the cells below.")

---

## Cell 2: Delegation Tokens -- Cryptographic Capability Control

In a multi-agent system, you cannot just let Agent B do whatever it wants because Agent A asked it to. **Delegation Tokens** solve this:

- Each token is **HMAC-SHA256 signed** -- tamper-proof
- Each token has **scoped capabilities** -- only the operations you explicitly grant
- Each token has an **expiry** -- time-bounded authority
- Each token is **bound to a task** -- cannot be reused for other purposes

Think of it as an OAuth token, but for AI agent-to-agent delegation.

In [ ]:
# === Delegation Tokens: Scoped, Signed, Time-Bounded Authority ===

print("=" * 70)
print("DELEGATION TOKENS -- Cryptographic Capability Control")
print("=" * 70)

# 1. Create a token: Scheduler delegates calendar reading to Agent B
print("\n--- Creating a delegation token ---")
print("Scenario: Scheduler grants CalendarAgent permission to read calendar")
print("          and send messages, but NOT write files or execute code.\n")

token = token_mgr.create_token(
    issuer="scheduler",
    subject="calendar-agent",
    ops_allowed=["read_calendar", "send_message"],
    binding=TokenBinding(task_id="briefing-2026-03-13", contract_id="morning-briefing-v1"),
    resource_selectors=[
        ResourceSelector(resource_type="calendar", resource_id="primary"),
        ResourceSelector(resource_type="messaging", resource_id="signal"),
    ],
    expiry_minutes=30,  # Expires in 30 minutes
)

# Show the token structure
print("Token structure:")
print(json.dumps({
    "token_id": token.token_id[:12] + "...",
    "issuer": token.issuer,
    "subject": token.subject,
    "ops_allowed": list(token.ops_allowed),
    "resources": [f"{r.resource_type}:{r.resource_id}" for r in token.resource_selectors],
    "expiry": token.expiry,
    "signature": token.signature[:24] + "...",
}, indent=2))

# Log the token creation
audit_log.append(
    intent_id="briefing-2026-03-13", task_id="token-creation",
    tuple_type="DCT",
    tuple_data={"action": "create", "issuer": token.issuer,
                "subject": token.subject, "ops": list(token.ops_allowed),
                "agent": "scheduler"},
)

# 2. Verify the token signature
print("\n--- Signature Verification ---")
valid, error = token_mgr.validate_token(token)
print(f"  Valid: {valid}")
print(f"  Error: {error}")
print(f"  Expired: {token.is_expired()}")

# 3. Check least privilege -- allowed operation
print("\n--- Least Privilege Enforcement ---")
allowed, err = token_mgr.check_least_privilege(token, "read_calendar")
print(f'  "read_calendar"  -> Allowed: {allowed}')

allowed, err = token_mgr.check_least_privilege(token, "send_message")
print(f'  "send_message"   -> Allowed: {allowed}')

# 4. Agent tries to exceed authority!
allowed, err = token_mgr.check_least_privilege(token, "write_file")
print(f'  "write_file"     -> Allowed: {allowed}  Error: {err}')

allowed, err = token_mgr.check_least_privilege(token, "execute_code")
print(f'  "execute_code"   -> Allowed: {allowed}  Error: {err}')

audit_log.append(
    intent_id="briefing-2026-03-13", task_id="token-validation",
    tuple_type="EVIDENCE",
    tuple_data={"action": "privilege_check", "agent": "calendar-agent",
                "attempted_ops": ["read_calendar", "send_message", "write_file", "execute_code"],
                "denied": ["write_file", "execute_code"]},
)

# 5. Demonstrate token expiry
print("\n--- Token Expiry ---")
expired_token = token_mgr.create_token(
    issuer="scheduler", subject="calendar-agent",
    ops_allowed=["read_calendar"],
    binding=TokenBinding(task_id="old-task", contract_id="old-contract"),
    expiry_minutes=-1,  # Already expired!
)
valid, error = token_mgr.validate_token(expired_token)
print(f"  Expired token valid: {valid}")
print(f"  Error code: {error}")

# 6. Demonstrate tamper detection
print("\n--- Tamper Detection ---")
tampered = DelegationCapabilityToken(
    token_id=token.token_id, issuer=token.issuer,
    subject="evil-agent",  # <-- Tampered!
    resource_selectors=token.resource_selectors, ops_allowed=("read_calendar", "write_file", "delete_all"),
    expiry=token.expiry, binding=token.binding, signature=token.signature,
)
valid, error = token_mgr.validate_token(tampered)
print(f"  Tampered token valid: {valid}")
print(f"  Error code: {error}")
print("  (Signature no longer matches -- tampering detected!)")

print("\n" + "=" * 70)
print(f"Audit log entries so far: {audit_log.count}")
print("=" * 70)

---

## Cell 3: Circuit Breaker -- Automatic Failure Recovery

When an AI agent calls an external API (Google Calendar, GitHub, Slack), that API might be down. Without a circuit breaker:

- Agent retries endlessly, burning tokens and money
- Failures cascade to other agents waiting on results
- The whole system grinds to a halt

The circuit breaker pattern prevents this with three states:

```
CLOSED (normal) --[failures >= threshold]--> OPEN (rejecting)
OPEN --[timeout elapsed]--> HALF_OPEN (testing)
HALF_OPEN --[probe succeeds]--> CLOSED
HALF_OPEN --[probe fails]--> OPEN
```

In [ ]:
# === Circuit Breaker: Containing Failures ===

print("=" * 70)
print("CIRCUIT BREAKER -- Automatic Failure Detection & Recovery")
print("=" * 70)

# Track state transitions for the demo
transitions = []

def log_transition(old, new):
    msg = f"  >> {old.name} -> {new.name}"
    transitions.append((old.name, new.name))
    print(msg)
    audit_log.append(
        intent_id="briefing-2026-03-13", task_id="circuit-breaker",
        tuple_type="SYSTEM",
        tuple_data={"action": "circuit_breaker_transition",
                    "old_state": old.name, "new_state": new.name,
                    "agent": "github-adapter"},
    )

# Simulated external API
call_count = 0
api_is_down = True

def github_api():
    global call_count
    call_count += 1
    if api_is_down:
        raise ConnectionError("GitHub API 503: Service Unavailable")
    return {"prs": 3, "issues": 7, "ci_status": "green"}

# Create circuit breaker with low threshold for demo
cb = CircuitBreaker(failure_threshold=3, recovery_timeout=2.0, on_state_change=log_transition)

print(f"\nInitial state: {cb.state.name}")
print(f"Failure threshold: 3 consecutive failures")
print(f"Recovery timeout: 2 seconds\n")

# Phase 1: API is down -- failures accumulate
print("--- Phase 1: API is down, failures accumulate ---")
for i in range(4):
    try:
        result = cb.call(github_api)
        print(f"  Call {i+1}: Success - {result}")
    except CircuitBreakerOpen as e:
        print(f"  Call {i+1}: REJECTED (breaker open, {e.failure_count} failures)")
    except ConnectionError as e:
        print(f"  Call {i+1}: Failed - {e}")

print(f"\nState: {cb.state.name} | Failures: {cb.failure_count}")
print(f"API calls made: {call_count} (breaker prevented call #{call_count + 1})")

# Phase 2: Wait for recovery timeout, then probe
print("\n--- Phase 2: Waiting for recovery timeout... ---")
time.sleep(2.5)  # Wait for recovery_timeout
print(f"State after timeout: {cb.state.name}")

# API still down -- probe fails
print("\n--- Phase 3: Probe attempt (API still down) ---")
try:
    cb.call(github_api)
except ConnectionError:
    print(f"  Probe failed -- back to {cb.state.name}")

# Phase 4: API recovers
print("\n--- Phase 4: API recovers ---")
api_is_down = False
time.sleep(2.5)  # Wait for recovery timeout again

try:
    result = cb.call(github_api)
    print(f"  Probe succeeded! Result: {result}")
    print(f"  State: {cb.state.name}")
except Exception as e:
    print(f"  Unexpected: {e}")

# Normal operation resumes
print("\n--- Phase 5: Normal operation resumed ---")
for i in range(3):
    result = cb.call(github_api)
    print(f"  Call: Success - {result}")

print(f"\nFinal state: {cb.state.name}")
print(f"Total API calls: {call_count}")
print(f"Transitions: {' -> '.join(t[1] for t in transitions)}")

print("\n" + "=" * 70)
print(f"Audit log entries so far: {audit_log.count}")
print("=" * 70)

---

## Cell 4: Kill Switch -- Graduated Emergency Halt

Sometimes you need to stop everything. But "everything" is nuanced:

| Mode | What Happens |
|------|-------------|
| `DISENGAGED` | Normal operation |
| `HALT_NONCRITICAL` | Non-critical tasks queued, critical tasks continue |
| `HALT_ALL` | All new work stopped, only safety-critical tasks allowed |
| `EMERGENCY` | Immediate halt, everything blocked, preserve state |

Critical tasks (safety monitoring, audit logging, cost tracking) survive all modes except EMERGENCY.

In [ ]:
# === Kill Switch: Graduated Emergency Response ===

print("=" * 70)
print("KILL SWITCH -- Graduated Emergency Halt")
print("=" * 70)

# Define the task types we'll test
tasks = [
    ("briefing_generation", "Non-critical"),
    ("calendar_sync", "Non-critical"),
    ("safety_monitoring", "CRITICAL"),
    ("audit_logging", "CRITICAL"),
    ("cost_tracking", "CRITICAL"),
    ("sprint_planning", "Non-critical"),
]

def check_all_tasks(label):
    print(f"\n  {'Task':<25} {'Type':<15} {'Allowed':<10} {'Action'}")
    print(f"  {'-'*25} {'-'*15} {'-'*10} {'-'*10}")
    for task, criticality in tasks:
        result = kill_switch.check_task_allowed(task)
        status = "YES" if result["allowed"] else "NO"
        print(f"  {task:<25} {criticality:<15} {status:<10} {result['action']}")

# Mode 1: DISENGAGED (normal)
print("\n--- Mode: DISENGAGED (normal operation) ---")
print(f"Current mode: {kill_switch.mode.name}")
check_all_tasks("DISENGAGED")

# Mode 2: HALT_NONCRITICAL
print("\n\n--- Engaging: HALT_NONCRITICAL ---")
event = kill_switch.engage(
    mode=KillSwitchMode.HALT_NONCRITICAL,
    reason="Budget warning: 80% of daily limit consumed",
    triggered_by="cost_governor",
    affected_tasks=3,
)
print(f"Event: {event.mode.name} at {event.timestamp}")
print(f"Reason: {event.reason}")
check_all_tasks("HALT_NONCRITICAL")

# Mode 3: HALT_ALL
print("\n\n--- Escalating: HALT_ALL ---")
event = kill_switch.engage(
    mode=KillSwitchMode.HALT_ALL,
    reason="Budget exceeded: $50/$40 daily limit",
    triggered_by="cost_governor",
    affected_tasks=5,
)
print(f"Event: {event.mode.name}")
print(f"Reason: {event.reason}")
check_all_tasks("HALT_ALL")

# Mode 4: EMERGENCY
print("\n\n--- Escalating: EMERGENCY ---")
event = kill_switch.engage(
    mode=KillSwitchMode.EMERGENCY,
    reason="Security incident: unauthorized agent detected",
    triggered_by="security_monitor",
    affected_tasks=6,
)
print(f"Event: {event.mode.name}")
print(f"Reason: {event.reason}")
check_all_tasks("EMERGENCY")
print("\n  (In EMERGENCY mode, even critical tasks are blocked!)")

# Reset
print("\n\n--- Disengaging kill switch ---")
event = kill_switch.disengage(
    triggered_by="human_operator",
    reason="Incident resolved, resuming operations",
)
print(f"Mode: {kill_switch.mode.name}")
print(f"Reason: {event.reason}")

# Show event history
print("\n--- Kill Switch Event History ---")
for i, evt in enumerate(kill_switch.get_history()):
    marker = ">>" if evt.mode != KillSwitchMode.DISENGAGED else "<<"
    print(f"  {marker} {evt.mode.name:<20} by {evt.triggered_by:<20} | {evt.reason}")

print("\n" + "=" * 70)
print(f"Audit log entries so far: {audit_log.count}")
print("=" * 70)

---

## Cell 5: Governance Audit Log -- Complete Accountability

Every action from the previous cells was **automatically logged** to an append-only audit trail. This is not optional -- the governance bus enforces it.

The audit log supports:
- **HMAC-signed entries** -- tamper-evident
- **Query by agent, action type, or intent** -- instant forensics
- **Daily rotation with 180-day retention** -- EU AI Act compliant
- **Append-only guarantee** -- entries can never be modified or deleted

In [ ]:
# === Governance Audit Log: Everything Was Recorded ===

print("=" * 70)
print("GOVERNANCE AUDIT LOG -- Append-Only Accountability")
print("=" * 70)

# 1. Total entries
print(f"\nTotal audit entries: {audit_log.count}")
print(f"Log location: {_DEMO_DIR / 'governance'}")

# 2. Browse all entries
print("\n--- Complete Audit Trail ---")
print(f"{'#':<4} {'Type':<12} {'Task':<22} {'Action':<30} {'Signature'}")
print(f"{'-'*4} {'-'*12} {'-'*22} {'-'*30} {'-'*16}")
for i, entry in enumerate(audit_log.entries):
    action = entry.tuple_data.get("action", "--")
    sig = entry.signature[:16] + "..." if entry.signature else "UNSIGNED"
    print(f"{i+1:<4} {entry.tuple_type:<12} {entry.task_id:<22} {action:<30} {sig}")

# 3. Query by type
print("\n--- Query: All DCT (token) events ---")
dct_entries = audit_log.query(tuple_type="DCT")
for e in dct_entries:
    print(f"  [{e.timestamp}] {e.tuple_data.get('action')}: "
          f"{e.tuple_data.get('issuer')} -> {e.tuple_data.get('subject')}")

print("\n--- Query: All SYSTEM events ---")
sys_entries = audit_log.query(tuple_type="SYSTEM")
for e in sys_entries:
    action = e.tuple_data.get('action', '')
    if 'kill_switch' in action:
        mode = e.tuple_data.get('mode', '')
        reason = e.tuple_data.get('reason', '')
        print(f"  [{e.timestamp}] Kill switch -> {mode}: {reason}")
    elif 'circuit_breaker' in action:
        old = e.tuple_data.get('old_state', '')
        new = e.tuple_data.get('new_state', '')
        print(f"  [{e.timestamp}] Circuit breaker: {old} -> {new}")

# 4. Query by intent
print("\n--- Query: All events for briefing-2026-03-13 ---")
intent_entries = audit_log.query(intent_id="briefing-2026-03-13")
print(f"  Found {len(intent_entries)} entries for this briefing run")

# 5. Show the raw JSONL on disk
print("\n--- Raw JSONL on disk (first entry) ---")
log_files = list((_DEMO_DIR / "governance").glob("*.jsonl"))
if log_files:
    with open(log_files[0]) as f:
        first_line = f.readline().strip()
    parsed = json.loads(first_line)
    print(json.dumps(parsed, indent=2)[:500] + "...")

# 6. Integrity verification
print("\n--- Integrity Verification ---")
all_signed = all(e.signature for e in audit_log.entries)
print(f"  All entries signed: {all_signed}")
print(f"  Entries with HMAC-SHA256 signatures: {sum(1 for e in audit_log.entries if e.signature)}")
print(f"  Append-only: entries can never be modified or deleted")

print("\n" + "=" * 70)
print(f"Audit log entries: {audit_log.count}")
print("=" * 70)

---

## Cell 6: Delegation Chains -- Bounded Subdelegation

What happens when Agent A delegates to Agent B, and Agent B wants to delegate to Agent C? Without bounds, delegation chains can grow infinitely:

```
Human -> Agent A -> Agent B -> Agent C -> Agent D -> Agent E -> ...
```

Each hop loses context, increases cost, and reduces accountability. **Delegation Chains** enforce:

- **Maximum chain depth** (default: 3) -- hard limit on subdelegation
- **State machine transitions** -- every delegation follows a defined lifecycle
- **Replan budgets** -- failed tasks can retry, but only N times
- **Budget inheritance** -- child tasks inherit (and can only narrow) parent budgets

In [ ]:
# === Delegation Chains: Bounded Subdelegation ===

print("=" * 70)
print("DELEGATION CHAINS -- Bounded Subdelegation")
print("=" * 70)

# 1. Create root delegation: Human -> Scheduler
print("\n--- Building a delegation chain ---")
print("  Max chain depth: 3\n")

root = DelegationContext(
    intent_id="morning-briefing",
    task_id=str(uuid.uuid4()),
    delegator_id="human",
    delegatee_id="scheduler",
    contract_id="briefing-contract-v1",
    chain_depth=0,
    budget=DelegationBudget(max_tokens=100000, max_cost_usd=5.0, max_wall_time_seconds=300),
)

audit_log.append(
    intent_id=root.intent_id, task_id=root.task_id,
    tuple_type="DCTX",
    tuple_data={"action": "delegate", "agent": "human",
                "from": root.delegator_id, "to": root.delegatee_id,
                "depth": root.chain_depth},
)

print(f"  Depth 0: {root.delegator_id} -> {root.delegatee_id} (root)")

# 2. Scheduler delegates to CalendarAgent (depth 1)
child1, err = root.create_child("calendar-agent", "calendar-read-v1")
print(f"  Depth 1: {child1.delegator_id} -> {child1.delegatee_id}")

audit_log.append(
    intent_id=child1.intent_id, task_id=child1.task_id,
    tuple_type="DCTX",
    tuple_data={"action": "delegate", "agent": "scheduler",
                "from": child1.delegator_id, "to": child1.delegatee_id,
                "depth": child1.chain_depth},
)

# 3. CalendarAgent delegates to GoogleAPI adapter (depth 2)
child2, err = child1.create_child("google-api-adapter", "gcal-api-v1")
print(f"  Depth 2: {child2.delegator_id} -> {child2.delegatee_id}")

audit_log.append(
    intent_id=child2.intent_id, task_id=child2.task_id,
    tuple_type="DCTX",
    tuple_data={"action": "delegate", "agent": "calendar-agent",
                "from": child2.delegator_id, "to": child2.delegatee_id,
                "depth": child2.chain_depth},
)

# 4. GoogleAPI adapter delegates to OAuth handler (depth 3 = max)
child3, err = child2.create_child("oauth-handler", "oauth-refresh-v1")
print(f"  Depth 3: {child3.delegator_id} -> {child3.delegatee_id} (max depth!)")

audit_log.append(
    intent_id=child3.intent_id, task_id=child3.task_id,
    tuple_type="DCTX",
    tuple_data={"action": "delegate", "agent": "google-api-adapter",
                "from": child3.delegator_id, "to": child3.delegatee_id,
                "depth": child3.chain_depth},
)

# 5. OAuth handler tries to delegate further -- BLOCKED!
print("\n--- Depth enforcement ---")
child4, err = child3.create_child("token-refresher", "refresh-v1")
print(f"  Depth 4 attempt: {child4}")
print(f"  Error: {err}")
print(f"  (Chain depth {child3.chain_depth + 1} > max {DEFAULT_MAX_CHAIN_DEPTH} -- BLOCKED!)")

audit_log.append(
    intent_id=root.intent_id, task_id="depth-violation",
    tuple_type="EVIDENCE",
    tuple_data={"action": "delegation_denied", "agent": "oauth-handler",
                "attempted_depth": child3.chain_depth + 1,
                "max_depth": DEFAULT_MAX_CHAIN_DEPTH,
                "error": err},
)

# 6. State machine transitions
print("\n--- State Machine Lifecycle ---")
states = []
for new_state in ["ISSUED", "RUNNING", "EVIDENCE_READY", "VERIFIED"]:
    ok, err = child2.transition(new_state)
    status = "OK" if ok else f"FAIL ({err})"
    states.append(new_state)
    print(f"  {' -> '.join(['PROPOSED'] + states)}: {status}")

# 7. Invalid transition
print("\n--- Invalid transition attempt ---")
ok, err = child2.transition("RUNNING")
print(f"  VERIFIED -> RUNNING: {'OK' if ok else f'BLOCKED ({err})'}")
print(f"  (VERIFIED is a terminal state -- no going back!)")

# 8. Replan budget
print("\n--- Replan Budget ---")
replan_ctx = DelegationContext(
    intent_id="replan-demo", task_id=str(uuid.uuid4()),
    delegator_id="scheduler", delegatee_id="github-agent",
    contract_id="github-read-v1", chain_depth=0, status="PROPOSED",
)

# Walk through: PROPOSED -> ISSUED -> RUNNING -> EVIDENCE_READY -> REPLANNED
for s in ["ISSUED", "RUNNING", "EVIDENCE_READY"]:
    replan_ctx.transition(s)

print(f"  Max replans: {DEFAULT_MAX_REPLANS}")
for attempt in range(DEFAULT_MAX_REPLANS + 1):
    ok, err = replan_ctx.transition("REPLANNED")
    if ok:
        print(f"  Replan #{replan_ctx.replan_count}: Allowed (replans used: {replan_ctx.replan_count}/{DEFAULT_MAX_REPLANS})")
        # Reset to EVIDENCE_READY for next attempt
        replan_ctx.transition("PROPOSED")
        for s in ["ISSUED", "RUNNING", "EVIDENCE_READY"]:
            replan_ctx.transition(s)
    else:
        print(f"  Replan #{attempt + 1}: DENIED -- {err}")
        print(f"  (Budget exhausted: {replan_ctx.replan_count}/{DEFAULT_MAX_REPLANS} replans used)")

print("\n" + "=" * 70)
print(f"Audit log entries so far: {audit_log.count}")
print("=" * 70)

---

## Cell 7: Full Scenario -- Morning Briefing Pipeline

Now let's put it all together in a realistic scenario. This is how HUMMBL's Morning Briefing system actually works:

1. **Scheduler** delegates to 5 specialized agents
2. Each agent gets a **scoped token** with only the capabilities it needs
3. One agent hits a **circuit breaker** (external API down)
4. Budget overrun triggers the **kill switch**
5. The **audit log** captures the complete trail

This is governance in action -- not as overhead, but as infrastructure that makes multi-agent systems *safe to deploy*.

In [ ]:
# === Full Scenario: Morning Briefing Pipeline ===

print("=" * 70)
print("FULL SCENARIO -- Morning Briefing Pipeline")
print("=" * 70)

# Fresh instances for the scenario
scenario_audit = GovernanceBus(base_dir=_DEMO_DIR / "scenario")
scenario_ks = KillSwitchCore()
scenario_ks.subscribe(lambda evt: scenario_audit.append(
    intent_id="briefing-scenario", task_id="kill_switch",
    tuple_type="SYSTEM", tuple_data={"action": "kill_switch", "mode": evt.mode.name,
                                     "reason": evt.reason, "agent": evt.triggered_by},
))

# Define the agent fleet
agents = [
    {"id": "calendar-agent",  "ops": ["read_calendar"],           "resource": "calendar"},
    {"id": "github-agent",    "ops": ["read_prs", "read_ci"],     "resource": "github"},
    {"id": "linear-agent",    "ops": ["read_issues"],             "resource": "linear"},
    {"id": "cost-agent",      "ops": ["read_costs", "alert"],     "resource": "billing"},
    {"id": "security-agent",  "ops": ["run_scan", "read_alerts"], "resource": "security"},
]

intent_id = "briefing-2026-03-13-full"
total_cost = 0.0
BUDGET_LIMIT = 2.50

# Step 1: Create delegation tokens for each agent
print("\n[Step 1] Creating delegation tokens for 5 agents...")
tokens = {}
for agent in agents:
    t = token_mgr.create_token(
        issuer="scheduler", subject=agent["id"],
        ops_allowed=agent["ops"],
        binding=TokenBinding(task_id=intent_id, contract_id=f"{agent['resource']}-contract-v1"),
        resource_selectors=[ResourceSelector(resource_type=agent["resource"])],
        expiry_minutes=15,
    )
    tokens[agent["id"]] = t
    scenario_audit.append(
        intent_id=intent_id, task_id=f"delegate-{agent['id']}",
        tuple_type="DCT",
        tuple_data={"action": "create_token", "agent": "scheduler",
                    "subject": agent["id"], "ops": agent["ops"]},
    )
    print(f"  [token] {agent['id']:<20} ops={agent['ops']}")

# Step 2: Agents execute their tasks
print("\n[Step 2] Agents executing tasks...")

# Circuit breakers per external service
breakers = {}
for agent in agents:
    breakers[agent["id"]] = CircuitBreaker(failure_threshold=2, recovery_timeout=5.0)

# Simulate API calls with varying results
api_results = {
    "calendar-agent":  {"success": True, "data": "3 meetings today", "cost": 0.02},
    "github-agent":    {"success": False, "error": "GitHub API 503", "cost": 0.01},
    "linear-agent":    {"success": True, "data": "7 open issues", "cost": 0.03},
    "cost-agent":      {"success": True, "data": "$2.47 spent today", "cost": 0.01},
    "security-agent":  {"success": True, "data": "No alerts", "cost": 0.05},
}

for agent in agents:
    aid = agent["id"]
    result = api_results[aid]
    cb = breakers[aid]

    # Check kill switch first
    ks_result = scenario_ks.check_task_allowed(aid)
    if not ks_result["allowed"]:
        print(f"  [BLOCKED] {aid:<20} Kill switch: {ks_result['reason']}")
        scenario_audit.append(
            intent_id=intent_id, task_id=f"exec-{aid}",
            tuple_type="EVIDENCE",
            tuple_data={"action": "blocked", "agent": aid, "reason": "kill_switch"},
        )
        continue

    # Validate token
    valid, err = token_mgr.validate_token(tokens[aid])
    if not valid:
        print(f"  [DENIED] {aid:<20} Token invalid: {err}")
        continue

    # Execute through circuit breaker
    if result["success"]:
        def success_fn(r=result):
            return r["data"]
        try:
            data = cb.call(success_fn)
            total_cost += result["cost"]
            print(f"  [  OK  ] {aid:<20} Result: {data:<25} Cost: ${result['cost']:.2f}  Total: ${total_cost:.2f}")
            scenario_audit.append(
                intent_id=intent_id, task_id=f"exec-{aid}",
                tuple_type="EVIDENCE",
                tuple_data={"action": "success", "agent": aid, "data": data, "cost": result["cost"]},
            )
        except CircuitBreakerOpen:
            print(f"  [CIRCUIT] {aid:<20} Circuit breaker OPEN")
    else:
        # Simulate 2 failures to trip the breaker
        for attempt in range(3):
            try:
                def fail_fn(r=result):
                    raise ConnectionError(r["error"])
                cb.call(fail_fn)
            except CircuitBreakerOpen:
                total_cost += result["cost"] * attempt  # Failed calls still cost
                print(f"  [CIRCUIT] {aid:<20} OPEN after {attempt} failures: {result['error']}")
                scenario_audit.append(
                    intent_id=intent_id, task_id=f"exec-{aid}",
                    tuple_type="EVIDENCE",
                    tuple_data={"action": "circuit_breaker_open", "agent": aid,
                                "failures": attempt, "error": result["error"]},
                )
                break
            except ConnectionError:
                pass

# Step 3: Budget check triggers kill switch
print(f"\n[Step 3] Budget check: ${total_cost:.2f} / ${BUDGET_LIMIT:.2f}")

if total_cost > BUDGET_LIMIT * 0.8:
    print(f"  [WARNING] Budget at {total_cost/BUDGET_LIMIT*100:.0f}% -- engaging HALT_NONCRITICAL")
    scenario_ks.engage(
        mode=KillSwitchMode.HALT_NONCRITICAL,
        reason=f"Budget warning: ${total_cost:.2f}/${BUDGET_LIMIT:.2f} ({total_cost/BUDGET_LIMIT*100:.0f}%)",
        triggered_by="cost_governor",
        affected_tasks=5,
    )
else:
    print(f"  [OK] Budget within limits ({total_cost/BUDGET_LIMIT*100:.0f}%)")

# Step 4: Show the complete audit trail
print(f"\n[Step 4] Complete audit trail ({scenario_audit.count} entries)")
print(f"{'#':<4} {'Type':<10} {'Agent':<22} {'Action':<25}")
print(f"{'-'*4} {'-'*10} {'-'*22} {'-'*25}")
for i, entry in enumerate(scenario_audit.entries):
    agent = entry.tuple_data.get('agent', entry.tuple_data.get('subject', '--'))
    action = entry.tuple_data.get('action', '--')
    print(f"{i+1:<4} {entry.tuple_type:<10} {str(agent):<22} {action:<25}")

# Step 5: Cleanup
if scenario_ks.engaged:
    scenario_ks.disengage(triggered_by="demo", reason="Demo complete")

print("\n" + "=" * 70)
print("SCENARIO COMPLETE")
print(f"  Agents deployed:     {len(agents)}")
print(f"  Tokens issued:       {len(tokens)}")
print(f"  Circuit breakers:    {sum(1 for cb in breakers.values() if cb.state == CircuitBreakerState.OPEN)} tripped")
print(f"  Kill switch events:  {len(scenario_ks.get_history())}")
print(f"  Audit trail entries: {scenario_audit.count}")
print(f"  Total cost:          ${total_cost:.2f}")
print("=" * 70)

---

## What You Just Saw

In 5 minutes, you experienced the five governance primitives that make multi-agent AI systems safe to deploy:

| Primitive | What It Does | Why It Matters |
|---|---|---|
| **Delegation Tokens** | HMAC-signed, scoped, time-bounded authority | Agents cannot exceed their delegated capabilities |
| **Circuit Breaker** | Automatic failure detection and recovery | Failures are contained, not cascaded |
| **Kill Switch** | Graduated emergency halt (4 modes) | Humans retain ultimate control |
| **Governance Bus** | Append-only, signed audit log | Complete accountability and compliance |
| **Delegation Chains** | Bounded subdelegation with depth limits | Prevents runaway agent delegation |

### Key Properties

- **Zero dependencies** -- pure Python stdlib, runs anywhere
- **Production-tested** -- 5,000+ tests in the founder-mode platform
- **EU AI Act ready** -- 180-day audit retention, signed entries
- **Thread-safe** -- all primitives are safe for concurrent use
- **Fail-closed** -- security defaults to deny, not allow

---

### Try It Yourself

Modify the parameters above and re-run:
- Change `failure_threshold` in the circuit breaker -- what happens with threshold=1 vs threshold=10?
- Set `expiry_minutes=0` on a token -- can an agent use it?
- Change `DEFAULT_MAX_CHAIN_DEPTH` to 2 -- which delegation gets blocked?
- Change the budget limit in the scenario -- does the kill switch fire?

---

### Learn More

**HUMMBL Governance-as-a-Service**

- [hummbl.io](https://hummbl.io) -- GaaS platform and Base120 API
- [github.com/hummbl-dev](https://github.com/hummbl-dev) -- Open source governance tools
- [Base120](https://hummbl.io) -- 120 mental models for AI governance

Built by HUMMBL LLC. Governance for the agents that govern your business.